# SnapCal Local Training Notebook

This notebook runs the shortest reliable local workflow for SnapCal on your own machine. It assumes you already have the repo locally, trains one raw model per run based on `MODEL_NAME` (`resnet50`, `efficientnet_b0`, or `vit_b16`), skips segmentation, and exports a model-specific bundle for the Django API.

Before you run it:
- Start Jupyter from inside the SnapCal repo if possible.
- Make sure your local Python/Jupyter environment can install PyTorch and friends.
- Download Food-101 locally first and point `FOOD101_ROOT` at it, or place it under `data/raw`.
- Make sure your `food101_usda_mapping.csv` has been curated before you trust calorie metrics.
- This notebook prefers Apple MPS on macOS when available, otherwise it will use CPU.


In [9]:
REPO_DIR = None  # Optional absolute path to the SnapCal repo root. Leave as None to auto-detect.
DATA_ROOT = None  # Optional absolute path for downloaded Food-101 data. Defaults to REPO_DIR/data/raw
EXPORT_ROOT = None  # Optional absolute path for copied outputs. Defaults to REPO_DIR/artifacts/exports/<RUN_NAME>
MODEL_NAME = "vit_b16"  # Choose from: resnet50, efficientnet_b0, vit_b16
SUPPORTED_MODELS = {"resnet50", "efficientnet_b0", "vit_b16"}
if MODEL_NAME not in SUPPORTED_MODELS:
    raise ValueError(f"Unsupported MODEL_NAME: {MODEL_NAME}. Choose from {sorted(SUPPORTED_MODELS)}")
RUN_NAME = f"snapcal_{MODEL_NAME}_local_v1"
FOOD101_ROOT = "/Users/jaymalave/Desktop/SnapCal/data/raw/food-101"  # Set this to your local Food-101 dataset root if you already know it
EXPERIMENT_NAME = f"{MODEL_NAME}_raw_local"
BASE_TRAINING_CONFIG = f"configs/training/{MODEL_NAME}_raw.json"
LOCAL_TRAINING_CONFIG = f"artifacts/configs/{EXPERIMENT_NAME}.json"
EXPORT_CONFIG = LOCAL_TRAINING_CONFIG
EXPORT_CHECKPOINT = f"artifacts/models/{EXPERIMENT_NAME}/best.pt"
EXPORT_BUNDLE_DIR = f"artifacts/models/production_bundle_{MODEL_NAME}"
DEFAULT_BATCH_SIZES = {
    "resnet50": 16,
    "efficientnet_b0": 16,
    "vit_b16": 8,
}
LOCAL_BATCH_SIZE = DEFAULT_BATCH_SIZES[MODEL_NAME]
LOCAL_EPOCHS = 3
LOCAL_NUM_WORKERS = 0


In [10]:
import json
import os
from pathlib import Path
import shlex
import shutil
import subprocess
import sys

os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

def run_checked(args, cwd=None, display_args=None):
    args = [str(arg) for arg in args]
    visible_args = args if display_args is None else [str(arg) for arg in display_args]
    printable = ' '.join(shlex.quote(arg) for arg in visible_args)
    print(f'$ {printable}')
    subprocess.run(args, check=True, cwd=None if cwd is None else str(cwd))

def resolve_repo_dir(explicit_repo_dir=None):
    if explicit_repo_dir:
        repo_dir = Path(explicit_repo_dir).expanduser().resolve()
        expected = repo_dir / 'ml' / 'scripts' / 'train.py'
        if not expected.exists():
            raise FileNotFoundError(f'REPO_DIR does not look like SnapCal: {repo_dir}')
        return repo_dir
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'ml' / 'scripts' / 'train.py').exists():
            return candidate
    raise FileNotFoundError('Could not auto-detect the SnapCal repo root. Set REPO_DIR explicitly and rerun this cell.')

def detect_food101_root(search_root):
    search_root = Path(search_root)
    matches = []
    for train_file in search_root.rglob('meta/train.txt'):
        candidate = train_file.parent.parent
        if (candidate / 'meta' / 'test.txt').exists() and (candidate / 'images').exists():
            matches.append(candidate)
    if not matches:
        raise FileNotFoundError(
            f'Could not find a Food-101 dataset root under {search_root}. ' 
            'Expected a directory containing both meta/train.txt and images/.'
        )
    unique_matches = sorted(set(matches), key=lambda path: (len(path.parts), str(path)))
    return unique_matches[0]

repo_path = resolve_repo_dir(REPO_DIR)
REPO_DIR = str(repo_path)
DATA_ROOT = str(Path(DATA_ROOT).expanduser().resolve()) if DATA_ROOT else str(repo_path / 'data' / 'raw')
EXPORT_ROOT = str(Path(EXPORT_ROOT).expanduser().resolve()) if EXPORT_ROOT else str(repo_path / 'artifacts' / 'exports' / RUN_NAME)
Path(DATA_ROOT).mkdir(parents=True, exist_ok=True)
Path(EXPORT_ROOT).mkdir(parents=True, exist_ok=True)
print(f'Repo dir: {REPO_DIR}')
print(f'Data root: {DATA_ROOT}')
print(f'Export root: {EXPORT_ROOT}')


Repo dir: /Users/jaymalave/Desktop/SnapCal
Data root: /Users/jaymalave/Desktop/SnapCal/data/raw
Export root: /Users/jaymalave/Desktop/SnapCal/artifacts/exports/snapcal_vit_b16_local_v1


In [11]:
run_checked([sys.executable, '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel'], display_args=['python', '-m', 'pip', 'install', '-U', 'pip', 'setuptools', 'wheel'])
run_checked([
    sys.executable, '-m', 'pip', 'install',
    'numpy>=1.26,<3.0',
    'Pillow>=10.4,<11.0',
    'matplotlib>=3.8,<4.0',
    'pandas>=2.2,<3.0',
    'scikit-learn>=1.5,<2.0',
    'transformers>=4.45,<5.0',
], display_args=['python', '-m', 'pip', 'install', 'numpy>=1.26,<3.0', 'Pillow>=10.4,<11.0', 'matplotlib>=3.8,<4.0', 'pandas>=2.2,<3.0', 'scikit-learn>=1.5,<2.0', 'transformers>=4.45,<5.0'])

try:
    import torch
    import torchvision
except ImportError:
    run_checked([sys.executable, '-m', 'pip', 'install', 'torch>=2.4,<3.0', 'torchvision>=0.19,<1.0'], display_args=['python', '-m', 'pip', 'install', 'torch>=2.4,<3.0', 'torchvision>=0.19,<1.0'])
    import torch
    import torchvision

run_checked([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'], cwd=REPO_DIR, display_args=['python', '-m', 'pip', 'install', '-e', '.', '--no-deps'])

mps_available = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
if torch.cuda.is_available():
    selected_device = 'cuda'
elif mps_available:
    selected_device = 'mps'
else:
    selected_device = 'cpu'

print(f'torch={torch.__version__}')
print(f'torchvision={torchvision.__version__}')
print(f'selected device: {selected_device}')


$ python -m pip install -U pip setuptools wheel
Defaulting to user installation because normal site-packages is not writeable
$ python -m pip install 'numpy>=1.26,<3.0' 'Pillow>=10.4,<11.0' 'matplotlib>=3.8,<4.0' 'pandas>=2.2,<3.0' 'scikit-learn>=1.5,<2.0' 'transformers>=4.45,<5.0'
Defaulting to user installation because normal site-packages is not writeable
$ python -m pip install -e . --no-deps
Defaulting to user installation because normal site-packages is not writeable
Obtaining file:///Users/jaymalave/Desktop/SnapCal
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): fini

In [12]:
existing_root = None
if FOOD101_ROOT and Path(FOOD101_ROOT).exists():
    existing_root = Path(FOOD101_ROOT).expanduser().resolve()
else:
    try:
        existing_root = detect_food101_root(DATA_ROOT)
    except FileNotFoundError:
        existing_root = None

if existing_root is None:
    raise FileNotFoundError(
        'Food-101 was not found locally. Download it first and either set FOOD101_ROOT to the dataset root ' 
        'or place it somewhere under DATA_ROOT. Expected a directory containing meta/train.txt, meta/test.txt, and images/.'
    )

FOOD101_ROOT = str(existing_root)
print(f'Resolved FOOD101_ROOT = {FOOD101_ROOT}')
for path in sorted(Path(FOOD101_ROOT).iterdir()):
    print(path)


Resolved FOOD101_ROOT = /Users/jaymalave/Desktop/SnapCal/data/raw/food-101
/Users/jaymalave/Desktop/SnapCal/data/raw/food-101/.DS_Store
/Users/jaymalave/Desktop/SnapCal/data/raw/food-101/README.txt
/Users/jaymalave/Desktop/SnapCal/data/raw/food-101/images
/Users/jaymalave/Desktop/SnapCal/data/raw/food-101/license_agreement.txt
/Users/jaymalave/Desktop/SnapCal/data/raw/food-101/meta


In [13]:
from pathlib import Path

config_path = Path(REPO_DIR) / LOCAL_TRAINING_CONFIG
config_path.parent.mkdir(parents=True, exist_ok=True)
effective_batch_size = LOCAL_BATCH_SIZE if selected_device != 'cpu' else min(LOCAL_BATCH_SIZE, 8)
model_output_dir = f'artifacts/models/{EXPERIMENT_NAME}'
model_report_dir = f'artifacts/reports/{EXPERIMENT_NAME}'
override = {
    'extends': str((Path(REPO_DIR) / BASE_TRAINING_CONFIG).resolve()),
    'experiment_name': EXPERIMENT_NAME,
    'batch_size': effective_batch_size,
    'epochs': LOCAL_EPOCHS,
    'num_workers': LOCAL_NUM_WORKERS,
    'output_dir': model_output_dir,
    'report_dir': model_report_dir,
}
config_path.write_text(json.dumps(override, indent=2), encoding='utf-8')
print(f'Wrote local training config to: {config_path}')
print(json.dumps(override, indent=2))


Wrote local training config to: /Users/jaymalave/Desktop/SnapCal/artifacts/configs/vit_b16_raw_local.json
{
  "extends": "/Users/jaymalave/Desktop/SnapCal/configs/training/vit_b16_raw.json",
  "experiment_name": "vit_b16_raw_local",
  "batch_size": 8,
  "epochs": 3,
  "num_workers": 0,
  "output_dir": "artifacts/models/vit_b16_raw_local",
  "report_dir": "artifacts/reports/vit_b16_raw_local"
}


In [14]:
import time

manifest_path = Path(REPO_DIR) / 'data' / 'reference' / 'manifests' / 'food101_manifest.csv'
print(f'Manifest will be written to: {manifest_path}')
print(f'Using device: {selected_device}')

print('1/2 Building manifest...')
t0 = time.time()
run_checked([sys.executable, 'ml/scripts/build_manifest.py', '--dataset-root', FOOD101_ROOT, '--output', 'data/reference/manifests/food101_manifest.csv'], cwd=REPO_DIR, display_args=['python', 'ml/scripts/build_manifest.py', '--dataset-root', FOOD101_ROOT, '--output', 'data/reference/manifests/food101_manifest.csv'])
print(f'Manifest step finished in {(time.time() - t0) / 60:.2f} min')

if not manifest_path.exists():
    raise FileNotFoundError(f'Manifest was not created: {manifest_path}')

print('2/2 Running smoke test...')
t0 = time.time()
run_checked([sys.executable, 'ml/scripts/smoke_test.py', '--config', LOCAL_TRAINING_CONFIG], cwd=REPO_DIR, display_args=['python', 'ml/scripts/smoke_test.py', '--config', LOCAL_TRAINING_CONFIG])
print(f'Smoke test finished in {(time.time() - t0) / 60:.2f} min')
print('Manifest + smoke test are ready. This notebook intentionally skips segmentation for the faster local raw-model workflow.')


Manifest will be written to: /Users/jaymalave/Desktop/SnapCal/data/reference/manifests/food101_manifest.csv
Using device: mps
1/2 Building manifest...
$ python ml/scripts/build_manifest.py --dataset-root /Users/jaymalave/Desktop/SnapCal/data/raw/food-101 --output data/reference/manifests/food101_manifest.csv
Wrote 101000 manifest rows to data/reference/manifests/food101_manifest.csv
Manifest step finished in 0.03 min
2/2 Running smoke test...
$ python ml/scripts/smoke_test.py --config artifacts/configs/vit_b16_raw_local.json


/Users/jaymalave/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Smoke test logits shape: (4, 101)
Smoke test finished in 0.94 min
Manifest + smoke test are ready. This notebook intentionally skips segmentation for the faster local raw-model workflow.


## Training Run

This notebook trains one raw model per run. Set `MODEL_NAME` in the first code cell, then rerun from the top. If a previous run saved `artifacts/models/<MODEL_NAME>_raw_local/last.pt`, training will resume from it.


In [15]:
import time

config_file = Path(REPO_DIR) / LOCAL_TRAINING_CONFIG
if not config_file.exists():
    raise FileNotFoundError(f'Missing config file: {config_file}')

model_output_dir = Path(REPO_DIR) / 'artifacts' / 'models' / EXPERIMENT_NAME
last_checkpoint = model_output_dir / 'last.pt'
best_checkpoint = model_output_dir / 'best.pt'
print(f'Training config: {LOCAL_TRAINING_CONFIG}')
print(f'Using device: {selected_device}')
if last_checkpoint.exists():
    print(f'Resuming from checkpoint: {last_checkpoint}')
else:
    print(f'Starting a fresh {EXPERIMENT_NAME} training run')

t0 = time.time()
run_checked([sys.executable, 'ml/scripts/train.py', '--config', LOCAL_TRAINING_CONFIG], cwd=REPO_DIR, display_args=['python', 'ml/scripts/train.py', '--config', LOCAL_TRAINING_CONFIG])
print(f'Training finished in {(time.time() - t0) / 3600:.2f} hours')
print(f'best exists: {best_checkpoint.exists()} -> {best_checkpoint}')


Training config: artifacts/configs/vit_b16_raw_local.json
Using device: mps
Starting a fresh vit_b16_raw_local training run
$ python ml/scripts/train.py --config artifacts/configs/vit_b16_raw_local.json


/Users/jaymalave/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Best checkpoint: artifacts/models/vit_b16_raw_local/best.pt
Report: artifacts/reports/vit_b16_raw_local/test_report.json
Predictions: artifacts/reports/vit_b16_raw_local/test_predictions.json
Training finished in 6.38 hours
best exists: True -> /Users/jaymalave/Desktop/SnapCal/artifacts/models/vit_b16_raw_local/best.pt


In [16]:
checkpoint_path = Path(REPO_DIR) / EXPORT_CHECKPOINT
bundle_dir = Path(REPO_DIR) / EXPORT_BUNDLE_DIR
if not checkpoint_path.exists():
    raise FileNotFoundError(
        f'Expected checkpoint was not found: {checkpoint_path}. ' 
        'Update EXPORT_CHECKPOINT if you exported a different training run.'
    )
print(f'Exporting bundle from: {checkpoint_path}')
print(f'Bundle output dir: {bundle_dir}')
run_checked([
    sys.executable,
    'ml/scripts/export_inference_bundle.py',
    '--checkpoint',
    EXPORT_CHECKPOINT,
    '--config',
    EXPORT_CONFIG,
    '--mapping',
    'data/reference/food101_usda_mapping.csv',
    '--output-dir',
    EXPORT_BUNDLE_DIR,
    '--model-version',
    RUN_NAME,
], cwd=REPO_DIR, display_args=['python', 'ml/scripts/export_inference_bundle.py', '--checkpoint', EXPORT_CHECKPOINT, '--config', EXPORT_CONFIG, '--mapping', 'data/reference/food101_usda_mapping.csv', '--output-dir', EXPORT_BUNDLE_DIR, '--model-version', RUN_NAME])
run_checked(['rsync', '-av', EXPORT_BUNDLE_DIR, f'{EXPORT_ROOT}/'], cwd=REPO_DIR)
run_checked(['rsync', '-av', 'artifacts/reports', f'{EXPORT_ROOT}/'], cwd=REPO_DIR)
run_checked(['rsync', '-av', 'data/reference/manifests', f'{EXPORT_ROOT}/'], cwd=REPO_DIR)


Exporting bundle from: /Users/jaymalave/Desktop/SnapCal/artifacts/models/vit_b16_raw_local/best.pt
Bundle output dir: /Users/jaymalave/Desktop/SnapCal/artifacts/models/production_bundle_vit_b16
$ python ml/scripts/export_inference_bundle.py --checkpoint artifacts/models/vit_b16_raw_local/best.pt --config artifacts/configs/vit_b16_raw_local.json --mapping data/reference/food101_usda_mapping.csv --output-dir artifacts/models/production_bundle_vit_b16 --model-version snapcal_vit_b16_local_v1
Exported inference bundle to artifacts/models/production_bundle_vit_b16
$ rsync -av artifacts/models/production_bundle_vit_b16 /Users/jaymalave/Desktop/SnapCal/artifacts/exports/snapcal_vit_b16_local_v1/
Transfer starting: 5 files
production_bundle_vit_b16/
production_bundle_vit_b16/metadata.json
production_bundle_vit_b16/model.pt
production_bundle_vit_b16/nutrition_mapping.csv
production_bundle_vit_b16/segmentation_config.json

sent 1030859571 bytes  received 114 bytes  252469859 bytes/sec
total size

## Use The Exported Bundle

After this finishes for the selected local raw-model workflow:
- Your bundle will be at `artifacts/models/production_bundle_<MODEL_NAME>`.
- A copy will also be under `artifacts/exports/<RUN_NAME>`.
- Point `SNAPCAL_MODEL_BUNDLE` at the bundle you want to serve.
- For the fastest raw-only API path, run Django locally with `SNAPCAL_ENABLE_SEGMENTATION=false`.
